# Attention From Scratch (NumPy) Scaled Dot Product Attention

In [3]:
import numpy as np

def softmax(x):
    exp_x = np.exp(x - np.max(x, axis=-1, keepdims=True))
    return exp_x / np.sum(exp_x, axis=-1, keepdims=True)

def scaled_dot_product_attention(Q, K, V):
    dk = K.shape[-1]

    scores = np.matmul(Q, K.T) / np.sqrt(dk)

    attention_weights = softmax(scores)

    output = np.matmul(attention_weights, V)

    return output, attention_weights


# Example input
X = np.array([
    [1, 0, 1, 0],
    [0, 2, 0, 2],
    [1, 1, 1, 1]
], dtype=np.float32)

# Random projection matrices
Wq = np.random.randn(4, 4)
Wk = np.random.randn(4, 4)
Wv = np.random.randn(4, 4)

# Generate Q, K, V
Q = X @ Wq
K = X @ Wk
V = X @ Wv

output, attention = scaled_dot_product_attention(Q, K, V)

print("Attention Weights:")
print(attention)

print("\nOutput:")
print(output)

Attention Weights:
[[0.04927693 0.76680385 0.18391922]
 [0.95208786 0.02286045 0.02505169]
 [0.24896109 0.60031054 0.15072837]]

Output:
[[ 1.64344921 -3.63589671  1.83666223  0.87659105]
 [-0.38149442 -1.309655    0.39256911  1.1168462 ]
 [ 1.19268065 -3.11990622  1.51589885  0.93120757]]


# **Attention From Scratch (MHA via Numpy)**


* One head → syntax
* Another → semantics
* Another → long dependencies


In [4]:
import numpy as np

class MultiHeadAttention:

    def __init__(self, d_model, num_heads):

        assert d_model % num_heads == 0

        self.d_model = d_model
        self.num_heads = num_heads
        self.depth = d_model // num_heads

        self.Wq = np.random.randn(d_model, d_model)
        self.Wk = np.random.randn(d_model, d_model)
        self.Wv = np.random.randn(d_model, d_model)

        self.Wo = np.random.randn(d_model, d_model)

    def split_heads(self, x):

        batch_size, seq_len, d_model = x.shape

        x = x.reshape(
            batch_size,
            seq_len,
            self.num_heads,
            self.depth
        )

        return x.transpose(0, 2, 1, 3)

    def softmax(self, x):

        exp_x = np.exp(x - np.max(x, axis=-1, keepdims=True))

        return exp_x / np.sum(exp_x, axis=-1, keepdims=True)

    def scaled_dot_product_attention(self, Q, K, V):

        dk = K.shape[-1]

        scores = np.matmul(Q, K.transpose(0,1,3,2)) / np.sqrt(dk)

        attention_weights = self.softmax(scores)

        output = np.matmul(attention_weights, V)

        return output

    def forward(self, x):

        Q = x @ self.Wq
        K = x @ self.Wk
        V = x @ self.Wv

        Q = self.split_heads(Q)
        K = self.split_heads(K)
        V = self.split_heads(V)

        attention = self.scaled_dot_product_attention(Q, K, V)

        attention = attention.transpose(0,2,1,3)

        batch_size, seq_len, _, _ = attention.shape

        concat_attention = attention.reshape(
            batch_size,
            seq_len,
            self.d_model
        )

        output = concat_attention @ self.Wo

        return output


# Example
x = np.random.randn(2, 5, 8)

mha = MultiHeadAttention(
    d_model=8,
    num_heads=2
)

out = mha.forward(x)

print(out.shape)

(2, 5, 8)


# **Pytorch Version**

In [5]:
import torch
import torch.nn as nn
import math

class SelfAttention(nn.Module):

    def __init__(self, embed_size):
        super().__init__()

        self.embed_size = embed_size

        self.Wq = nn.Linear(embed_size, embed_size)
        self.Wk = nn.Linear(embed_size, embed_size)
        self.Wv = nn.Linear(embed_size, embed_size)

    def forward(self, x):

        Q = self.Wq(x)
        K = self.Wk(x)
        V = self.Wv(x)

        scores = torch.matmul(Q, K.transpose(-2, -1))

        scores = scores / math.sqrt(self.embed_size)

        attention = torch.softmax(scores, dim=-1)

        out = torch.matmul(attention, V)

        return out


x = torch.randn(2, 5, 16)

model = SelfAttention(16)

out = model(x)

print(out.shape)

torch.Size([2, 5, 16])


In [1]:
import torch
import torch.nn as nn
import math

class MultiHeadAttention(nn.Module):

    def __init__(self, embed_dim, num_heads):

        super(MultiHeadAttention, self).__init__()

        assert embed_dim % num_heads == 0

        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads

        self.Wq = nn.Linear(embed_dim, embed_dim)
        self.Wk = nn.Linear(embed_dim, embed_dim)
        self.Wv = nn.Linear(embed_dim, embed_dim)

        self.fc_out = nn.Linear(embed_dim, embed_dim)

    def split_heads(self, x):

        batch_size, seq_len, embed_dim = x.shape

        x = x.view(
            batch_size,
            seq_len,
            self.num_heads,
            self.head_dim
        )

        return x.transpose(1, 2)

    def forward(self, x):

        batch_size = x.shape[0]

        Q = self.Wq(x)
        K = self.Wk(x)
        V = self.Wv(x)

        Q = self.split_heads(Q)
        K = self.split_heads(K)
        V = self.split_heads(V)

        scores = torch.matmul(
            Q,
            K.transpose(-2, -1)
        ) / math.sqrt(self.head_dim)

        attention_weights = torch.softmax(
            scores,
            dim=-1
        )

        attention_output = torch.matmul(
            attention_weights,
            V
        )

        attention_output = attention_output.transpose(1, 2)

        attention_output = attention_output.contiguous().view(
            batch_size,
            -1,
            self.embed_dim
        )

        output = self.fc_out(attention_output)

        return output


# Example
x = torch.randn(2, 5, 16)

mha = MultiHeadAttention(
    embed_dim=16,
    num_heads=4
)

output = mha(x)

print(output.shape)

torch.Size([2, 5, 16])
